Last edited: 1/28/2026, 3:38 PM CST
# Table of climate models and members that we use

In [3]:
# import base packages for data analysis
import pandas as pd
import xarray as xr

from config import DATA_ROOT
from src.utils import extract_model_name
from src.cmip_dataclass import CMIP6EnsembleConfig

In [5]:
# shared attributes
fit = 'nonstat'
data_types = ['raw', 'anom_trend', 'anom_annmean']
TMIN = 1979
era5_variable = 't2m_annual_max'
cmip_variable = 'tas_annual_max'

CMIPConfig = CMIP6EnsembleConfig.from_yaml(
    'config/meta.yaml',
    'config/qc.yaml'
)

In [6]:
# ERA5 analysis
ds_era5 = xr.open_dataset(DATA_ROOT / 'ERA5' / 'gev' / f'era5_{era5_variable}_1deg_landonly_gev_{fit}_TMIN{TMIN}.nc', 
                            engine='netcdf4')
ds_era5

<xarray.Dataset>
Dimensions:               (year: 46, lat: 180, lon: 360)
Coordinates:
  * lat                   (lat) float64 -89.5 -88.5 -87.5 ... 87.5 88.5 89.5
  * lon                   (lon) int64 0 1 2 3 4 5 6 ... 354 355 356 357 358 359
  * year                  (year) int64 1979 1980 1981 1982 ... 2022 2023 2024
Data variables: (12/21)
    t2m                   (year, lat, lon) float32 ...
    t2m_anom_annmean      (year, lat, lon) float32 ...
    t2m_anom_trend        (year, lat, lon) float64 ...
    loc_raw               (lat, lon) float64 ...
    loc_t_raw             (lat, lon) float64 ...
    scale_raw             (lat, lon) float64 ...
    ...                    ...
    loc_anom_trend        (lat, lon) float64 ...
    loc_t_anom_trend      (lat, lon) float64 ...
    scale_anom_trend      (lat, lon) float64 ...
    scale_t_anom_trend    (lat, lon) float64 ...
    shape_anom_trend      (lat, lon) float64 ...
    shape_t_anom_trend    (lat, lon) float64 ...

In [ ]:
for anom_type in data_types:
    # make file/model matcher for 
    data_path = DATA_ROOT / 'CMIP6' / cmip_variable / 'gev'

    # Make all landonly file names
    fnames = [f for f in data_path.glob(f"*{fit}*{anom_type}*.nc") if "allmems" not in f.name]  # screen out allmems results

    modelname_filepath_matcher = {
    extract_model_name(f): f for f in fnames
    }

In [4]:
rows = []

for m in CMIPConfig.iter_active_models('tas_annual_max'):
    tmp_prim_mem = m.primary_member
    prim_mem = tmp_prim_mem.split("_")[1]

    rows.append({
        "Model": m.name,
        "Variables": "tas_annual_max, tas_annual_mean",
        "Primary Member": prim_mem,
        "Time Interval": "1979-2024",
        "Experiment": "historical + SSP5-8.5"
    })

In [5]:
df = pd.DataFrame(rows)
df.sort_values("Model")

,Model,Variables,Primary Member,Time Interval,Experiment
0,AWI-CM-1-1-MR,"tas_annual_max, tas_annual_mean",r1i1p1f1,1979-2024,historical + SSP5-8.5
1,BCC-CSM2-MR,"tas_annual_max, tas_annual_mean",r1i1p1f1,1979-2024,historical + SSP5-8.5
2,CAMS-CSM1-0,"tas_annual_max, tas_annual_mean",r1i1p1f1,1979-2024,historical + SSP5-8.5
3,CESM2-WACCM,"tas_annual_max, tas_annual_mean",r1i1p1f1,1979-2024,historical + SSP5-8.5
4,CMCC-CM2-SR5,"tas_annual_max, tas_annual_mean",r1i1p1f1,1979-2024,historical + SSP5-8.5
5,CMCC-ESM2,"tas_annual_max, tas_annual_mean",r1i1p1f1,1979-2024,historical + SSP5-8.5
6,CNRM-CM6-1-HR,"tas_annual_max, tas_annual_mean",r1i1p1f1,1979-2024,historical + SSP5-8.5
7,CNRM-ESM2-1,"tas_annual_max, tas_annual_mean",r10i1p1f2,1979-2024,historical + SSP5-8.5
8,EC-Earth3-CC,"tas_annual_max, tas_annual_mean",r1i1p1f1,1979-2024,historical + SSP5-8.5
9,EC-Earth3-Veg,"tas_annual_max, tas_annual_mean",r10i1p1f1,1979-2024,historical + SSP5-8.5


In [6]:
df.to_csv(
    DATA_ROOT / 'CMIP6' / 'supplemental_info' / 'model_var_primmem_time_exp.csv',
    index=False
)